# Unsloth + OpenEnv GRPO (Autoscaling)

Minimal Colab notebook to train an autoscaling policy with **Unsloth + TRL GRPO** against the project OpenEnv-style HTTP environment.

In [ ]:
%%capture
!pip -q install --upgrade pip
!pip -q install unsloth trl transformers peft accelerate datasets fastapi uvicorn bitsandbytes xformers

In [ ]:
import os
import subprocess
import time
from pathlib import Path

REPO_URL = "<YOUR_REPO_URL>"
REPO_DIR = Path("/content/OpenEnv2")
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print("Working dir:", REPO_DIR)

In [ ]:
from client import OpenEnvClient

trace_path = Path("traces.jsonl")
if trace_path.exists():
    print(f"[trace-check] Found existing trace file: {trace_path}")
else:
    print(f"[trace-check] Missing {trace_path}. Generating traces now...")
    !python3 generate_traces.py --num-traces 30 --output traces.jsonl --episode-length 120 --seed 7
    print(f"[trace-check] Generated trace file: {trace_path}")

base_url = "http://127.0.0.1:8000"
server_proc = subprocess.Popen([
    "python3", "server.py", "--trace-path", "traces.jsonl", "--host", "127.0.0.1", "--port", "8000", "--seed", "7"
])

client = OpenEnvClient(base_url)
deadline = time.time() + 25
while time.time() < deadline:
    try:
        if client.health().ok:
            print("Environment is healthy")
            break
    except Exception:
        pass
    time.sleep(0.5)
else:
    raise RuntimeError("Environment server did not become healthy in time")

In [ ]:
!python3 colab_train_rl.py \
  --base-url http://127.0.0.1:8000 \
  --init-model sft_model_balanced \
  --output-dir rl_model_unsloth_grpo \
  --num-prompts 48 \
  --prompt-seed 7 \
  --max-steps 30 \
  --learning-rate 5e-6 \
  --batch-size 1 \
  --gradient-accumulation-steps 2 \
  --num-generations 4 \
  --max-prompt-length 512 \
  --max-completion-length 8 \
  --seed 7

In [ ]:
!python3 eval_policy.py --trace-path traces.jsonl --max-traces 6 --sft-model-path sft_model_balanced --rl-model-path rl_model_unsloth_grpo

In [ ]:
from pathlib import Path
print("Saved RL outputs:")
for p in sorted(Path("rl_model_unsloth_grpo").glob("*")):
    print("-", p)

In [ ]:
server_proc.terminate()
print("Server terminated")